# Chapter 7: Reinforcement Learning and Optimal Control
## Practical: Deep Q-Learning on CartPole

In [ ]:
import gym
import numpy as np
import random
from collections import deque
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Environment
env = gym.make('CartPole-v1')
state_dim = env.observation_space.shape[0]
n_actions = env.action_space.n

# Hyperparameters
lr = 1e-3
gamma = 0.99
batch_size = 64
buffer_capacity = 10000
epsilon_start = 1.0
epsilon_end = 0.01
epsilon_decay = 0.995
target_update_freq = 100
num_episodes = 500

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Neural Network for Q(s,a)

In [ ]:
class DQN(nn.Module):
    def __init__(self, state_dim, n_actions):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.out = nn.Linear(128, n_actions)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.out(x)

## Replay Buffer

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, transition):
        self.buffer.append(transition)

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        return zip(*batch)

    def __len__(self):
        return len(self.buffer)

## DQN Agent

In [ ]:
class DQNAgent:
    def __init__(self):
        self.q_net = DQN(state_dim, n_actions).to(device)
        self.target_net = DQN(state_dim, n_actions).to(device)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.target_net.eval()
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.memory = ReplayBuffer(buffer_capacity)
        self.epsilon = epsilon_start
        self.step_count = 0

    def select_action(self, state):
        if np.random.random() < self.epsilon:
            return np.random.randint(n_actions)
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            q_values = self.q_net(state_t)
        return q_values.argmax().item()

    def store_transition(self, state, action, reward, next_state, done):
        self.memory.push((state, action, reward, next_state, done))

    def update(self):
        if len(self.memory) < batch_size:
            return
        states, actions, rewards, next_states, dones = self.memory.sample(batch_size)
        
        states = torch.FloatTensor(np.array(states)).to(device)
        actions = torch.LongTensor(np.array(actions)).unsqueeze(1).to(device)
        rewards = torch.FloatTensor(np.array(rewards)).unsqueeze(1).to(device)
        next_states = torch.FloatTensor(np.array(next_states)).to(device)
        dones = torch.FloatTensor(np.array(dones)).unsqueeze(1).to(device)

        q_values = self.q_net(states).gather(1, actions)
        with torch.no_grad():
            max_next_q = self.target_net(next_states).max(1, keepdim=True)[0]
            target_q = rewards + gamma * max_next_q * (1 - dones)
        
        loss = nn.MSELoss()(q_values, target_q)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        self.epsilon = max(epsilon_end, self.epsilon * epsilon_decay)
        self.step_count += 1
        if self.step_count % target_update_freq == 0:
            self.target_net.load_state_dict(self.q_net.state_dict())

## Training Loop

In [ ]:
agent = DQNAgent()
episode_rewards = []

for episode in range(num_episodes):
    state = env.reset()[0]
    total_reward = 0
    done = False
    while not done:
        action = agent.select_action(state)
        next_state, reward, done, truncated, _ = env.step(action)
        done = done or truncated
        agent.store_transition(state, action, reward, next_state, done)
        agent.update()
        state = next_state
        total_reward += reward
    episode_rewards.append(total_reward)
    if (episode+1) % 50 == 0:
        avg_reward = np.mean(episode_rewards[-50:])
        print(f"Episode {episode+1}, Avg Reward (last 50): {avg_reward:.2f}, Epsilon: {agent.epsilon:.3f}")

# Plot learning curve
plt.figure(figsize=(10, 5))
plt.plot(episode_rewards)
plt.xlabel('Episode', fontsize=12)
plt.ylabel('Total Reward', fontsize=12)
plt.title('DQN on CartPole', fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

## Observations

- The agent learns to balance the pole after 100-200 episodes.
- Rewards increase from ~10-20 to the maximum of 500.
- Experience replay decorrelates samples, mimicking a stationary dataset.
- The target network provides stable targets for convergence.
- Epsilon-greedy exploration is gradually annealed (lowering temperature).